# 🏥 AI-Driven Medical Inventory Forecasting — Full EDA
> **Goal:** Understand the dataset structure, distributions, correlations, and patterns before building the ML pipeline.

**Sections:**
1. Setup & Data Loading
2. Schema Validation & Data Quality
3. Target Variable Analysis (stockout_event)
4. Demand Signal Analysis (Time Series)
5. Supply Chain Feature Analysis
6. Stock Position Analysis
7. Contextual & Categorical Features
8. Correlation & Feature Importance (Pre-model)
9. Derived Feature Engineering
10. EDA Summary & Recommendations

---
## 1. Setup & Data Loading

In [2]:
# ── Install dependencies (run once) ──────────────────────────────────────────
!pip install pandas numpy matplotlib seaborn scikit-learn scipy missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import missingno as msno
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ All imports successful')

✅ All imports successful


In [3]:
# ── Load your CSV ─────────────────────────────────────────────────────────────
# Replace the path below with your actual file path
DATA_PATH = 'medical_inventory.csv'   # ← change this

df = pd.read_csv(DATA_PATH, parse_dates=['snapshot_date'])

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["snapshot_date"].min().date()} → {df["snapshot_date"].max().date()}')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'medical_inventory.csv'

---
## 2. Schema Validation & Data Quality

In [ ]:
# ── Column catalogue ─────────────────────────────────────────────────────────
IDENTIFIERS = ['record_id','snapshot_date','facility_id','facility_type',
               'department','item_id','item_category','item_subcategory',
               'unit_of_measure','primary_vendor_id']

DEMAND_FEATURES = ['avg_daily_usage_last_30d','avg_daily_usage_last_90d',
                   'usage_cv_last_90d','demand_trend','seasonal_demand_factor',
                   'recent_usage_spike']

CONTEXTUAL_FEATURES = ['criticality_level','shelf_life_days',
                       'facility_census_pct','pandemic_or_surge_flag']

SUPPLY_CHAIN_FEATURES = ['vendor_reliability_score','contracted_lead_time_days',
                         'actual_avg_lead_time_last_6m','lead_time_variability_days',
                         'active_po_in_transit','backorder_frequency_last_12m',
                         'sole_source_item','substitution_available']

STOCK_FEATURES = ['current_stock_on_hand','safety_stock_level','days_of_supply_on_hand',
                  'stock_as_pct_of_safety_level','reorder_point_days',
                  'days_until_next_scheduled_order','days_since_last_stockout']

TARGET = ['stockout_event']

ALL_GROUPS = {
    'Identifiers': IDENTIFIERS,
    'Demand': DEMAND_FEATURES,
    'Contextual': CONTEXTUAL_FEATURES,
    'Supply Chain': SUPPLY_CHAIN_FEATURES,
    'Stock Position': STOCK_FEATURES,
    'Target': TARGET
}

for group, cols in ALL_GROUPS.items():
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    print(f'{group:15s}: {len(present):2d} present  {"❌ MISSING: " + str(missing) if missing else "✅"}')

In [ ]:
# ── Data types & nulls ───────────────────────────────────────────────────────
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().mean() * 100).round(2),
    'nunique': df.nunique(),
    'sample': df.iloc[0]
})
summary.style.background_gradient(subset=['null_pct'], cmap='OrRd')

In [ ]:
# ── Missing value visualisation ───────────────────────────────────────────────
msno.matrix(df, figsize=(16, 5), color=(0.25, 0.45, 0.75))
plt.title('Missing Value Matrix — white = missing', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
dupes = df.duplicated(subset=['snapshot_date','facility_id','department','item_id'])
print(f'Duplicate (date × facility × dept × item) rows: {dupes.sum():,}')

# Unique time series keys
ts_keys = df.groupby(['facility_id','department','item_id']).size()
print(f'Unique time series (item × dept × facility): {len(ts_keys):,}')
print(f'Snapshots per series — median: {ts_keys.median():.0f}, max: {ts_keys.max()}')

---
## 3. Target Variable Analysis — `stockout_event`

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
target_counts = df['stockout_event'].value_counts()
imbalance_ratio = target_counts[0] / target_counts.get(1, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts.plot(kind='bar', ax=axes[0], color=['#2196F3','#F44336'],
                   edgecolor='white', width=0.5)
axes[0].set_title('Stockout Event — Class Distribution')
axes[0].set_xticklabels(['No Stockout (0)','Stockout (1)'], rotation=0)
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}\n({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x()+0.25, p.get_height()+50), ha='center')

# Pie
axes[1].pie(target_counts, labels=['No Stockout','Stockout'],
            colors=['#2196F3','#F44336'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proportional Split')

plt.suptitle(f'Class Imbalance Ratio (0:1) = {imbalance_ratio:.1f}:1', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print(f'⚠️  Imbalance ratio: {imbalance_ratio:.1f}:1 — use class_weight="balanced" or SMOTE')

In [ ]:
# ── Stockout rate by criticality ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

stockout_by_crit = df.groupby('criticality_level')['stockout_event'].mean().sort_values(ascending=False)
stockout_by_crit.plot(kind='bar', ax=axes[0], color='#E53935', edgecolor='white')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title('Stockout Rate by Criticality Level')
axes[0].set_xlabel('')

stockout_by_cat = df.groupby('item_category')['stockout_event'].mean().sort_values(ascending=False)
stockout_by_cat.plot(kind='barh', ax=axes[1], color='#1565C0', edgecolor='white')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title('Stockout Rate by Item Category')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Stockout rate over time ───────────────────────────────────────────────────
stockout_ts = df.groupby('snapshot_date')['stockout_event'].mean()

plt.figure(figsize=(14, 4))
stockout_ts.plot(color='#E53935', linewidth=1.5)
stockout_ts.rolling(7).mean().plot(color='#B71C1C', linewidth=2.5, label='7-day MA')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.title('Stockout Rate Over Time')
plt.xlabel('Date')
plt.ylabel('Stockout Rate')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Demand Signal Analysis (Time Series)

In [ ]:
# ── Distribution of usage metrics ────────────────────────────────────────────
demand_cols = ['avg_daily_usage_last_30d','avg_daily_usage_last_90d','usage_cv_last_90d']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, col in enumerate(demand_cols):
    # Histogram
    axes[0, i].hist(df[col].dropna(), bins=50, color='#1565C0', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel(col)

    # Boxplot split by stockout
    df.boxplot(column=col, by='stockout_event', ax=axes[1, i],
               boxprops=dict(color='#1565C0'),
               medianprops=dict(color='#E53935', linewidth=2))
    axes[1, i].set_title(f'{col} by Stockout')
    axes[1, i].set_xlabel('stockout_event (0=No, 1=Yes)')

plt.suptitle('Demand Feature Distributions', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 30d vs 90d usage — drift detection ───────────────────────────────────────
df['demand_drift'] = df['avg_daily_usage_last_30d'] / df['avg_daily_usage_last_90d'].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(df['avg_daily_usage_last_90d'], df['avg_daily_usage_last_30d'],
                alpha=0.3, s=15, c=df['stockout_event'], cmap='coolwarm')
axes[0].plot([0, df['avg_daily_usage_last_90d'].max()],
             [0, df['avg_daily_usage_last_90d'].max()], 'k--', lw=1, label='No drift line')
axes[0].set_xlabel('Avg Daily Usage (90d)')
axes[0].set_ylabel('Avg Daily Usage (30d)')
axes[0].set_title('30d vs 90d Usage\n(red=stockout, blue=no stockout)')
axes[0].legend()

df['demand_drift'].clip(-1, 5).hist(bins=60, ax=axes[1], color='#7B1FA2', edgecolor='white')
axes[1].axvline(1.0, color='black', linestyle='--', label='No drift')
axes[1].set_title('Demand Drift Ratio (30d/90d)\n>1 = accelerating demand')
axes[1].set_xlabel('Drift Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Items with accelerating demand (ratio>1.2): {(df["demand_drift"]>1.2).sum():,} ({(df["demand_drift"]>1.2).mean()*100:.1f}%)')

In [ ]:
# ── Demand trend & seasonality ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

trend_counts = df['demand_trend'].value_counts()
trend_counts.plot(kind='bar', ax=axes[0], color='#00796B', edgecolor='white')
axes[0].set_title('Demand Trend Distribution')
axes[0].set_xticklabels(trend_counts.index, rotation=30)

df.boxplot(column='seasonal_demand_factor', by='demand_trend', ax=axes[1],
           boxprops=dict(color='#00796B'),
           medianprops=dict(color='#E53935', linewidth=2))
axes[1].set_title('Seasonal Demand Factor by Trend')
axes[1].set_xlabel('Demand Trend')

plt.tight_layout()
plt.show()

---
## 5. Supply Chain Feature Analysis

In [ ]:
# ── Lead time analysis ────────────────────────────────────────────────────────
df['lead_time_bias'] = df['actual_avg_lead_time_last_6m'] - df['contracted_lead_time_days']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Lead time bias distribution
df['lead_time_bias'].hist(bins=50, ax=axes[0], color='#E65100', edgecolor='white')
axes[0].axvline(0, color='black', linestyle='--', lw=1.5, label='On time')
axes[0].set_title('Lead Time Bias\n(Actual − Contracted days)')
axes[0].set_xlabel('Days')
axes[0].legend()

# Lead time variability vs stockout
df.boxplot(column='lead_time_variability_days', by='stockout_event', ax=axes[1],
           boxprops=dict(color='#E65100'),
           medianprops=dict(color='#1565C0', linewidth=2))
axes[1].set_title('Lead Time Variability by Stockout')
axes[1].set_xlabel('stockout_event')

# Vendor reliability vs stockout rate
df['vendor_reliability_bin'] = pd.cut(df['vendor_reliability_score'],
                                       bins=[0, 0.6, 0.75, 0.9, 1.01],
                                       labels=['Low (<0.6)','Medium (0.6-0.75)',
                                               'High (0.75-0.9)','Excellent (>0.9)'])
vr_stockout = df.groupby('vendor_reliability_bin', observed=True)['stockout_event'].mean()
vr_stockout.plot(kind='bar', ax=axes[2], color='#1565C0', edgecolor='white')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[2].set_title('Stockout Rate by Vendor Reliability')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=20)

plt.tight_layout()
plt.show()

pct_late = (df['lead_time_bias'] > 0).mean()
print(f'Vendors delivering LATE (actual > contracted): {pct_late*100:.1f}%')

In [ ]:
# ── Sole source & backorder analysis ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Sole source stockout rate
ss_rate = df.groupby('sole_source_item')['stockout_event'].mean()
ss_rate.plot(kind='bar', ax=axes[0], color=['#42A5F5','#E53935'], edgecolor='white')
axes[0].set_xticklabels(['Not Sole Source','Sole Source'], rotation=0)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title('Stockout Rate: Sole Source vs Not')

# Substitution availability
sub_rate = df.groupby('substitution_available')['stockout_event'].mean()
sub_rate.plot(kind='bar', ax=axes[1], color=['#E53935','#66BB6A'], edgecolor='white')
axes[1].set_xticklabels(['No Substitute','Has Substitute'], rotation=0)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title('Stockout Rate by Substitution Availability')

# Backorder frequency distribution
df['backorder_frequency_last_12m'].hist(bins=30, ax=axes[2],
                                         color='#7B1FA2', edgecolor='white')
axes[2].set_title('Backorder Frequency (last 12m)')
axes[2].set_xlabel('Number of backorder events')

plt.tight_layout()
plt.show()

---
## 6. Stock Position Analysis

In [ ]:
# ── Days of supply vs stockout ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# KDE by class
for label, color in [(0,'#1565C0'),(1,'#E53935')]:
    subset = df[df['stockout_event']==label]['days_of_supply_on_hand'].dropna()
    subset.plot.kde(ax=axes[0], label=f'stockout={label}', color=color, linewidth=2)
axes[0].set_title('Days of Supply — KDE by Stockout Class')
axes[0].set_xlabel('Days of Supply')
axes[0].set_xlim(left=-5)
axes[0].legend()
axes[0].axvline(0, color='black', linestyle='--', lw=1)

# Stock % vs safety level scatter
sample = df.sample(min(2000, len(df)), random_state=42)
scatter = axes[1].scatter(sample['stock_as_pct_of_safety_level'],
                          sample['days_until_next_scheduled_order'],
                          c=sample['stockout_event'], cmap='coolwarm',
                          alpha=0.5, s=15)
axes[1].axvline(100, color='black', linestyle='--', lw=1, label='Safety stock level')
axes[1].set_xlabel('Stock as % of Safety Level')
axes[1].set_ylabel('Days Until Next Order')
axes[1].set_title('Stock Position vs Replenishment Window\n(red=stockout)')
plt.colorbar(scatter, ax=axes[1], label='stockout_event')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Projected quantity (key derived feature) ──────────────────────────────────
df['projected_qty'] = (df['current_stock_on_hand']
                       - df['avg_daily_usage_last_30d'] * df['actual_avg_lead_time_last_6m'])

pct_negative = (df['projected_qty'] < 0).mean()
pct_neg_stockout = df[df['projected_qty'] < 0]['stockout_event'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['projected_qty'].clip(-200, 500).hist(bins=60, ax=axes[0],
                                          color='#00796B', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', lw=2, label='Zero stock line')
axes[0].set_title('Projected Quantity at Lead Time')
axes[0].set_xlabel('Units')
axes[0].legend()
axes[0].text(0.6, 0.8, f'{pct_negative*100:.1f}% items\nwill stock out',
             transform=axes[0].transAxes, color='red', fontsize=11)

df.boxplot(column='projected_qty', by='stockout_event', ax=axes[1],
           boxprops=dict(color='#00796B'),
           medianprops=dict(color='#E53935', linewidth=2))
axes[1].set_title('Projected Qty by Stockout')
axes[1].set_xlabel('stockout_event')
axes[1].axhline(0, color='red', linestyle='--', lw=1)

plt.tight_layout()
plt.show()

print(f'Items with projected_qty < 0: {pct_negative*100:.1f}%')
print(f'Stockout rate where projected_qty < 0: {pct_neg_stockout*100:.1f}%')

---
## 7. Contextual & Categorical Features

In [ ]:
# ── Stockout rate by department & facility ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

dept_stockout = df.groupby('department')['stockout_event'].mean().sort_values(ascending=False)
dept_stockout.plot(kind='barh', ax=axes[0], color='#1565C0', edgecolor='white')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title('Stockout Rate by Department')

fac_stockout = df.groupby('facility_type')['stockout_event'].mean().sort_values(ascending=False)
fac_stockout.plot(kind='bar', ax=axes[1], color='#E65100', edgecolor='white')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title('Stockout Rate by Facility Type')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ── Census & surge flag impact ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Facility census vs stockout
df['census_bin'] = pd.cut(df['facility_census_pct'],
                           bins=[0, 0.5, 0.7, 0.85, 1.01],
                           labels=['Low (<50%)','Medium (50-70%)',
                                   'High (70-85%)','Full (>85%)'])
census_stockout = df.groupby('census_bin', observed=True)['stockout_event'].mean()
census_stockout.plot(kind='bar', ax=axes[0], color='#7B1FA2', edgecolor='white')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title('Stockout Rate by Census Level')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)

# Pandemic flag
surge_rate = df.groupby('pandemic_or_surge_flag')['stockout_event'].mean()
surge_rate.plot(kind='bar', ax=axes[1],
                color=['#42A5F5','#E53935'], edgecolor='white')
axes[1].set_xticklabels(['Normal','Pandemic/Surge'], rotation=0)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title('Stockout Rate: Normal vs Pandemic/Surge')

plt.tight_layout()
plt.show()

---
## 8. Correlation & Pre-model Feature Importance

In [ ]:
# ── Correlation heatmap (numeric features only) ───────────────────────────────
numeric_cols = (DEMAND_FEATURES + SUPPLY_CHAIN_FEATURES
                + STOCK_FEATURES + ['stockout_event'])
numeric_cols = [c for c in numeric_cols if c in df.columns
                and pd.api.types.is_numeric_dtype(df[c])]

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Point-biserial correlation with target ────────────────────────────────────
from scipy.stats import pointbiserialr

correlations = {}
for col in numeric_cols:
    if col == 'stockout_event':
        continue
    valid = df[[col, 'stockout_event']].dropna()
    if len(valid) > 30:
        r, p = pointbiserialr(valid['stockout_event'], valid[col])
        correlations[col] = {'correlation': r, 'p_value': p}

corr_df = pd.DataFrame(correlations).T.sort_values('correlation')
colors = ['#E53935' if v < 0 else '#1565C0' for v in corr_df['correlation']]

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_df)*0.35)))
corr_df['correlation'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_title('Point-Biserial Correlation with stockout_event\n(blue=positive, red=negative)', fontsize=12)
ax.set_xlabel('Correlation Coefficient')

# Mark significant features
for i, (_, row) in enumerate(corr_df.iterrows()):
    if row['p_value'] < 0.05:
        ax.text(row['correlation'] + 0.005, i, '*', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('Top 5 features positively correlated with stockout:')
print(corr_df.tail(5)[['correlation','p_value']].to_string())
print('\nTop 5 features negatively correlated with stockout:')
print(corr_df.head(5)[['correlation','p_value']].to_string())

In [ ]:
# ── Quick Random Forest feature importance (pre-model signal check) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

feature_cols = [c for c in numeric_cols if c != 'stockout_event']
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['stockout_event'].fillna(0).astype(int)

rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = pd.Series(rf.feature_importances_,
                           index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(importance_df)*0.35)))
importance_df.plot(kind='barh', ax=ax, color='#00796B', edgecolor='white')
ax.set_title('Random Forest Feature Importance (pre-model signal check)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 10 most important features:')
print(importance_df.tail(10).sort_values(ascending=False).to_string())

---
## 9. Derived Feature Engineering

In [ ]:
# ── Compute all derived features from architecture spec ───────────────────────
df['trend_ratio']       = (df['avg_daily_usage_last_30d']
                           / df['avg_daily_usage_last_90d'].replace(0, np.nan))

df['projected_qty']     = (df['current_stock_on_hand']
                           - df['avg_daily_usage_last_30d'] * df['actual_avg_lead_time_last_6m'])

df['lead_time_bias']    = (df['actual_avg_lead_time_last_6m']
                           - df['contracted_lead_time_days'])

df['supply_risk_score'] = (df['backorder_frequency_last_12m']
                           * df['lead_time_variability_days']
                           / df['vendor_reliability_score'].clip(0.01))

# Additional derived features
df['days_to_stockout']  = (df['current_stock_on_hand']
                           / df['avg_daily_usage_last_30d'].replace(0, np.nan))

df['safety_buffer']     = df['days_to_stockout'] - df['actual_avg_lead_time_last_6m']

derived = ['trend_ratio','projected_qty','lead_time_bias',
           'supply_risk_score','days_to_stockout','safety_buffer']

print('Derived features computed ✅')
df[derived].describe().round(2)

In [ ]:
# ── Derived features vs stockout ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes_flat = axes.flatten()

for i, col in enumerate(derived):
    data = df[[col,'stockout_event']].dropna()
    data[col] = data[col].clip(data[col].quantile(0.01), data[col].quantile(0.99))
    data.boxplot(column=col, by='stockout_event', ax=axes_flat[i],
                 boxprops=dict(color='#1565C0'),
                 medianprops=dict(color='#E53935', linewidth=2))
    axes_flat[i].set_title(col)
    axes_flat[i].set_xlabel('stockout_event')

plt.suptitle('Derived Features vs Stockout Event', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

---
## 10. EDA Summary & Recommendations

In [ ]:
# ── Final summary statistics ──────────────────────────────────────────────────
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)

print(f'\n📦 Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'🗓  Date range: {df["snapshot_date"].min().date()} → {df["snapshot_date"].max().date()}')
print(f'🔑 Unique time series: {df.groupby(["facility_id","department","item_id"]).ngroups:,}')

print(f'\n🎯 TARGET')
print(f'   Stockout rate: {df["stockout_event"].mean()*100:.2f}%')
print(f'   Class imbalance ratio: {imbalance_ratio:.1f}:1')

print(f'\n📊 DEMAND')
print(f'   Items with accelerating demand (30d/90d > 1.2): {(df["trend_ratio"]>1.2).sum():,}')
print(f'   Usage CV > 0.5 (high variability): {(df["usage_cv_last_90d"]>0.5).sum():,}')

print(f'\n🚚 SUPPLY CHAIN')
print(f'   Vendors delivering late: {(df["lead_time_bias"]>0).mean()*100:.1f}%')
print(f'   Sole-source items: {df["sole_source_item"].sum():,}')

print(f'\n📉 STOCK POSITION')
print(f'   Items projected to stock out before delivery: {(df["projected_qty"]<0).sum():,}')
print(f'   Items below safety stock now: {(df["stock_as_pct_of_safety_level"]<100).sum():,}')

print(f'\n✅ RECOMMENDATIONS FOR ML PIPELINE')
print('   1. Use SMOTE or class_weight="balanced" (imbalanced classes)')
print('   2. Use projected_qty & safety_buffer as top engineered features')
print('   3. Time-based train/val/test split (no random shuffle)')
print('   4. Separate model for pandemic_or_surge_flag==1 rows')
print('   5. Use stratified CV by criticality_level')
print('   6. Evaluate with F1, AUROC, Precision@K (top-K alerts)')